# 02 - Feature Engineering
Demonstrates the raw -> clean -> engineered-features -> target pipeline, and explicitly checks for temporal leakage.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "04_Source_Code"))
import config
from utils import get_logger
logger = get_logger("notebook")
print("Project root:", config.PROJECT_ROOT)
print("Study region:", config.STUDY_REGION["name"])


In [ ]:
from generate_demo_data import generate_raw_demo_data
from preprocessing import clean_dataset, save_missing_value_report
from feature_engineering import add_engineered_features, generate_target

raw = generate_raw_demo_data(save=True)
raw.head()


## Cleaning

In [ ]:
missing_report = save_missing_value_report(raw, config.PROCESSED_DATA_DIR / "missing_value_report.csv")
clean = clean_dataset(raw)
missing_report[missing_report.missing_count > 0]


## Feature engineering (rolling means, fire history, NDVI change, temperature anomaly, days since rain)
Every rolling feature uses `.shift(1)` before rolling so the **current** week's own reading never leaks into its own 'recent history' feature -- see the module docstring for details.

In [ ]:
engineered = add_engineered_features(clean)
engineered[["grid_id","date","temperature","temperature_7day_mean","temperature_anomaly",
            "rainfall","rainfall_7day_sum","days_since_rain",
            "fire_count_previous_7_days","fire_count_previous_30_days"]].head(10)


## Target generation (leak-free)
`fire_target` = will a fire occur in this cell during the **next** prediction window? The raw same-week ignition flag is dropped from the feature set afterwards.

In [ ]:
final = generate_target(engineered)
print("Positive rate:", final[config.TARGET_COLUMN].mean())
final.to_csv(config.PROCESSED_DATA_DIR / "demo_wildfire_dataset.csv", index=False)
final.head()


## Predictors vs. Target
**Predictors** (`config.FEATURE_COLUMNS`) are everything known *before* the prediction date: weather, vegetation, terrain, and historical fire counts. The **target** (`fire_target`) is the thing we are trying to forecast: whether a fire will occur in that grid cell during the *next* prediction window. Predictors must never contain information from after the prediction date -- this is the temporal-leakage rule enforced throughout this module.